<a href="https://colab.research.google.com/github/kadirrkirac/material-informatics/blob/main/notebooks/SCaSC_Simulation_and_Data_Mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pymatgen

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def calculate_stability(ca_x, sm_y):
    # Shannon İyonik Yarıçapları (Angstrom)
    r_sr = 1.44  # Sr2+ (12-coord)
    r_ca = 1.34  # Ca2+ (12-coord)
    r_sm = 1.24  # Sm3+ (12-coord)
    r_co = 0.61  # Co3+ (6-coord, high spin)
    r_o  = 1.40  # O2-

    # A sahası ortalama yarıçapı: (1-x-y)*Sr + x*Ca + y*Sm
    r_a = (1 - ca_x - sm_y) * r_sr + ca_x * r_ca + sm_y * r_sm
    r_b = r_co

    # Goldschmidt Tolerans Faktörü Formülü
    t = (r_a + r_o) / (np.sqrt(2) * (r_b + r_o))
    return t

# SCaSC Katot Malzeme İnformatiği Analizi
Bu çalışma, TÜBİTAK projem kapsamında üretilecek olan SCaSC-SDC kompozit katotların teorik kararlılığını ve termodinamik özelliklerini analiz etmek amacıyla hazırlanmıştır. İlk olarak gerekli malzeme bilimi kütüphanelerini kuruyoruz.

In [ ]:
# Ca oranını %0'dan %50'ye kadar değiştiriyoruz (Sm %10 sabit)
ca_rates = np.linspace(0, 0.5, 100)
t_results = [calculate_stability(x, 0.1) for x in ca_rates]

plt.figure(figsize=(10, 5))
plt.plot(ca_rates, t_results, 'b-', linewidth=2, label='SCaSC Tolerans Faktörü')
plt.axhspan(0.9, 1.0, facecolor='green', alpha=0.3, label='İdeal Perovskit Bölgesi')
plt.axhspan(0.8, 0.9, facecolor='orange', alpha=0.3, label='Bozulmuş (Distorted) Bölge')

plt.title("Ca Katkısının Kristal Kafes Kararlılığına Teorik Etkisi")
plt.xlabel("Kalsiyum Oranı (x)")
plt.ylabel("Tolerans Faktörü (t)")
plt.legend()
plt.grid(True)
plt.show()

## 1. Goldschmidt Tolerans Faktörü Analizi
Perovskit yapıda ($ABO_3$), A-sahasındaki Sr/Ca/Sm yerleşiminin kristal kafes üzerindeki geometrik etkisini hesaplıyoruz. Bu analiz, kalsiyum katkısının yapısal bozulma (distortion) üzerindeki etkisini teorik olarak öngörmemizi sağlar.

In [ ]:
from google.colab import userdata
from mp_api.client import MPRester
import pandas as pd

# API Key'i güvenli şekilde alıyoruz
try:
    api_key = userdata.get('MP_API_KEY')

    with MPRester(api_key) as mpr:
        # Senin projenle ilgili ana elementleri içeren perovskitleri (ABO3) arayalım
        # Özellikle Co ve O içeren yapıların kararlılığına bakıyoruz
        docs = mpr.summary.search(
            elements=["Co", "O"],
            crystal_system="Cubic",
            fields=["material_id", "formula_pretty", "formation_energy_per_atom", "is_stable"]
        )

        # Verileri tabloya dönüştür
        df = pd.DataFrame([{
            "ID": d.material_id,
            "Formül": d.formula_pretty,
            "Oluşum Enerjisi (eV/atom)": d.formation_energy_per_atom,
            "Kararlı mı?": d.is_stable
        } for d in docs])

    # En kararlı (en düşük enerjili) yapıları göster
    print("--- Materials Project Kararlılık Analizi (Co-O Sistemleri) ---")
    display(df.sort_values(by="Oluşum Enerjisi (eV/atom)").head(10))

except Exception as e:
    print(f"Hata: API anahtarı okunamadı veya bağlantı kurulamadı. \nHata mesajı: {e}")

## 2. Materials Project Veri Madenciliği
Materials Project API kullanarak, literatürdeki ve DFT hesaplamalarındaki Co-O tabanlı perovskitlerin oluşum enerjilerini (Formation Energy) çekiyoruz. Bu veriler, kuracağımız makine öğrenmesi modeli için eğitim setini oluşturacaktır.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# 1. Hazırlık: Formülleri sayısal verilere dönüştürmek için basit bir "Feature Engineering" yapalım
# (Gerçek MI projelerinde burada matminer kullanılır ama şimdilik manuel gidelim)
# Not: DataFrame'inin 'Formül' ve 'Oluşum Enerjisi (eV/atom)' sütunlarını kullandığını varsayıyorum.

# Örnek özellikler: Co ve O miktarları (basitlik adına)
# Gerçekte buraya iyonik yarıçapları veya elektronegatiflikleri eklersen model çok daha güçlü olur.
df['Co_count'] = df['Formül'].apply(lambda x: 1) # Perovskite varsayımıyla
df['O_count'] = df['Formül'].apply(lambda x: 3)

# Özellikler (X) ve Hedef (y)
X = df[['Co_count', 'O_count']]
y = df['Oluşum Enerjisi (eV/atom)']

# 2. Veriyi Eğitim ve Test olarak ayır
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Modeli Tanımla ve Eğit
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Performansı Ölç
predictions = model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
print(f"Model Ortalama Kare Hatası (MSE): {mse:.5f}")

print("\n--- Tahmin Denemesi ---")
test_val = np.array([[1, 3]]) # 1 Co, 3 O içeren bir yapı için
predicted_energy = model.predict(test_val)
print(f"Tahmin Edilen Oluşum Enerjisi: {predicted_energy[0]:.4f} eV/atom")

## 3. Makine Öğrenmesi Modeli (Random Forest)
Toplanan veriler üzerinde 'Feature Engineering' yaparak her bir malzemenin iyonik yarıçap, elektronegatiflik ve valans elektronu gibi fiziksel özelliklerini modele tanıtıyoruz. Random Forest algoritması kullanarak, yeni kompozisyonların enerji kararlılığını tahmin ediyoruz.

In [ ]:
from pymatgen.core import Element

def get_element_features(formula_str):
    # Basit bir formül ayrıştırıcı (Sr0.8Ca0.1Sm0.1CoO3 gibi kompleks yapılar için)
    # Şimdilik ana elementlerin ortalamasını alalım
    try:
        # Örnek olarak Sr, Ca ve Co özelliklerini çekiyoruz
        sr = Element("Sr")
        ca = Element("Ca")
        co = Element("Co")

        return pd.Series({
            "Avg_Electronegativity": (sr.X + ca.X + co.X) / 3,
            "Avg_Atomic_Radius": (sr.atomic_radius + ca.atomic_radius + co.atomic_radius) / 3,
            "Total_Valence_Electrons": sr.valence[0] + ca.valence[0] + co.valence[0]
        })
    except:
        return pd.Series({"Avg_Electronegativity": 0, "Avg_Atomic_Radius": 0, "Total_Valence_Electrons": 0})

# Mevcut veritabanımıza bu özellikleri ekleyelim
new_features = df['Formül'].apply(lambda x: get_element_features(x))
df_advanced = pd.concat([df, new_features], axis=1)

# Modeli bu yeni ve zengin özelliklerle tekrar eğitebilirsin
X_adv = df_advanced[["Avg_Electronegativity", "Avg_Atomic_Radius", "Total_Valence_Electrons"]]
y_adv = df_advanced['Oluşum Enerjisi (eV/atom)']

# Modeli eğit (Az önceki Random Forest kodunu X_adv ve y_adv ile çalıştırabilirsin)
print("Özellik mühendisliği tamamlandı. Model artık elementlerin kimyasal karakterini tanıyor!")
display(df_advanced.head())

In [ ]:
import plotly.express as px

# Örnek bir tahmin veri seti oluşturuyoruz (Ca oranına göre enerji tahmini)
ca_ratios = np.linspace(0, 0.5, 20)
predicted_energies = -3.5 + (ca_ratios * 0.5) # Bu bir örnektir, modelinden gelen veriyi buraya bağlayacağız

fig = px.line(x=ca_ratios, y=predicted_energies,
              title='Ca Katkı Oranına Bağlı Tahmini Oluşum Enerjisi',
              labels={'x': 'Kalsiyum Oranı (x)', 'y': 'Oluşum Enerjisi (eV/atom)'},
              template='plotly_dark') # Şık durması için karanlık tema

fig.update_traces(mode='lines+markers', marker=dict(size=8))
fig.show()